In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IVGMM
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. SETUP & DATA LOADING
# ==============================================================================
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
master_file = PROCESSED_DIR / "master_panel_data.csv"

df = pd.read_csv(master_file)
df = df.set_index(['Country Name', 'Year']).sort_index()

BRICS = ["Brazil", "Russia", "India", "China", "Egypt", "United Arab Emirates"]
OECD_EM = ["Chile", "Colombia", "Costa Rica", "Turkey", "Hungary", "Poland"]

def get_bloc(c):
    if c in BRICS: return 'BRICS (Developing)'
    if c in OECD_EM: return 'OECD (Emerging)'
    return 'OECD (Advanced)'

df['Economic_Bloc'] = [get_bloc(c) for c, y in df.index]

# ==============================================================================
# 2. ELITE CAPTURE RATIO & GMM DIFFERENCING (OVERIDENTIFIED)
# ==============================================================================
TOP_10_COL = 'Wealth Inequality (Top 10%)'
BOTTOM_50_COL = 'Wealth Inequality (Bottom 50%)'

# Create the Elite Capture Ratio
df['Wealth_Ratio_Top10_to_Bottom50'] = df[TOP_10_COL] / df[BOTTOM_50_COL]
df.replace([np.inf, -np.inf], np.nan, inplace=True)

Y_VAR = 'Wealth_Ratio_Top10_to_Bottom50' 
DEBT_VAR = 'General Government Debt (% of GDP)'
CONTROLS = ['GDP growth (annual %)', 'Gross capital formation (% of GDP)', 'Inflation (Annual %)']

# --- Create OVERIDENTIFIED Lags (t-2 AND t-3) ---
df['Inst_Ratio_Lag2'] = df.groupby(level=0)[Y_VAR].shift(2)
df['Inst_Debt_Lag2'] = df.groupby(level=0)[DEBT_VAR].shift(2)
df['Inst_Ratio_Lag3'] = df.groupby(level=0)[Y_VAR].shift(3)
df['Inst_Debt_Lag3'] = df.groupby(level=0)[DEBT_VAR].shift(3)

df[f'D_{Y_VAR}'] = df.groupby(level=0)[Y_VAR].diff(1)
df[f'D_{DEBT_VAR}'] = df.groupby(level=0)[DEBT_VAR].diff(1)
for var in CONTROLS: 
    df[f'D_{var}'] = df.groupby(level=0)[var].diff(1)

df[f'D_{Y_VAR}_Lag1'] = df.groupby(level=0)[f'D_{Y_VAR}'].shift(1)

# ==============================================================================
# 3. EXCEL-STYLE FORMATTER
# ==============================================================================
def print_excel_summary(res, title):
    print(f"\nSUMMARY OUTPUT: {title}")
    print(f"Obs: {res.nobs} (GMM lacks standard R-squared)")
    print("-" * 120)
    print(f"{'Variable':<40} {'Coeff':<15} {'Std.Err':<15} {'t-Stat':<10} {'P-val':<10}")
    params, se, t, p = res.params, res.std_errors, res.tstats, res.pvalues
    for var in params.index:
        print(f"{var[:38]:<40} {params[var]:<15.6f} {se[var]:<15.6f} {t[var]:<10.4f} {p[var]:<10.4f}")
    print("-" * 120)

# ==============================================================================
# 4. GMM EXECUTION FUNCTION
# ==============================================================================
def run_ratio_gmm(data, bloc_name):
    
    # Require all 4 instruments to exist for overidentification
    model_vars = [f'D_{Y_VAR}', f'D_{Y_VAR}_Lag1', f'D_{DEBT_VAR}'] + \
                 [f'D_{v}' for v in CONTROLS] + \
                 ['Inst_Ratio_Lag2', 'Inst_Debt_Lag2', 'Inst_Ratio_Lag3', 'Inst_Debt_Lag3']
                 
    reg_data = data[model_vars].dropna()
    
    if len(reg_data) < 20:
        print(f"\nSkipping GMM for {bloc_name}: Insufficient degrees of freedom after deep lagging.")
        return
    
    Y = reg_data[f'D_{Y_VAR}']
    X_exog = sm.add_constant(reg_data[[f'D_{v}' for v in CONTROLS]])
    X_endog = reg_data[[f'D_{Y_VAR}_Lag1', f'D_{DEBT_VAR}']]
    
    # OVERIDENTIFIED INSTRUMENTS MATRIX (4 variables)
    Instruments = reg_data[['Inst_Ratio_Lag2', 'Inst_Debt_Lag2', 'Inst_Ratio_Lag3', 'Inst_Debt_Lag3']]
    
    try:
        gmm_model = IVGMM(Y, X_exog, X_endog, Instruments, weight_type='robust')
        gmm_res = gmm_model.fit()
        print(f"\n[HANSEN J-TEST] {bloc_name} | p-value: {gmm_res.j_stat.pval:.4f}")
        print_excel_summary(gmm_res, f"Difference GMM (Wealth Ratio): {bloc_name}")
    except Exception as e:
        print(f"\nGMM Failed for {bloc_name} ({e})")

# ==============================================================================
# 5. PIPELINE EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # 1. RUN GLOBAL BASELINE (All Countries Combined)
    run_ratio_gmm(df, "GLOBAL PANEL (ALL COUNTRIES)")
    
    # 2. RUN INDIVIDUAL BLOCS
    for bloc in ['OECD (Advanced)', 'OECD (Emerging)', 'BRICS (Developing)']:
        df_bloc = df[df['Economic_Bloc'] == bloc]
        run_ratio_gmm(df_bloc, bloc)


[HANSEN J-TEST] GLOBAL PANEL (ALL COUNTRIES) | p-value: 0.8783

SUMMARY OUTPUT: Difference GMM (Wealth Ratio): GLOBAL PANEL (ALL COUNTRIES)
Obs: 1030 (GMM lacks standard R-squared)
------------------------------------------------------------------------------------------------------------------------
Variable                                 Coeff           Std.Err         t-Stat     P-val     
const                                    -0.728530       1.035571        -0.7035    0.4817    
D_GDP growth (annual %)                  0.240089        0.306934        0.7822     0.4341    
D_Gross capital formation (% of GDP)     0.455337        0.628957        0.7240     0.4691    
D_Inflation (Annual %)                   -0.298818       0.310512        -0.9623    0.3359    
D_Wealth_Ratio_Top10_to_Bottom50_Lag1    -0.161031       0.078345        -2.0554    0.0398    
D_General Government Debt (% of GDP)     0.089271        0.837818        0.1066     0.9151    
--------------------------------